## Utils

In [54]:
import asyncio
from typing import Any, Dict, List

import nest_asyncio

from langsmith import Client
from langsmith.evaluation import evaluate
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from run_pipeline import StaticParams
from src.all_functionality import load_eval_tasks
from src.evaluator import Evaluator

# Allow nested event loops (LangSmith runs its own loop)
nest_asyncio.apply()

# Global setup
EXPERIMENT_PREFIX = "Offline-Notion-Eval"
DATASET_NAME = "Eval_Run_Batch_1"
JUDGE_MODEL_NAME = "gemma27"

client = Client()
core_evaluator = Evaluator(default_judge_model=JUDGE_MODEL_NAME)
eval_tasks = load_eval_tasks("evals")


def _example_to_dict(example: Any) -> Dict[str, Any]:
    if isinstance(example, dict):
        return example
    out: Dict[str, Any] = {}
    out["inputs"] = getattr(example, "inputs", {}) or {}
    out["outputs"] = getattr(example, "outputs", {}) or {}
    return out


def _extract_task_and_state(example_like: Any) -> tuple[str, Dict[str, Any]]:
    ex = _example_to_dict(example_like)
    outputs = ex.get("outputs", {})
    state = outputs.get("pre_computed_state", {})
    task_id = outputs.get("task_id") or ex.get("inputs", {}).get("task_id", "")
    return task_id, state


def _present_score(status_items: List[Dict[str, Any]]) -> tuple[float, int]:
    present_count = sum(
        1
        for item in status_items
        if str(item.get("status", "")).strip().lower() == "present"
    )
    score = (present_count / len(status_items)) if status_items else 0.0
    return score, present_count


class RagStatementsEvaluator:
    """Independent evaluator for statement presence in RAG retrieval context."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        retrieval_context = state.get("retrieval_context", "")
        statements = self.task_specs.get(task_id, {}).get("correct_statements", [])

        status_items = await self.evaluator.eval_context_statements(
            context=retrieval_context,
            statements=statements,
            judge_model_name=self.judge_model_name,
        )

        score, present_count = _present_score(status_items)

        return {
            "key": "rag_statements_score",
            "score": score,
            "comment": f"task={task_id} present={present_count}/{len(status_items)}",
            "metadata": {"raw": status_items},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


class CodeStatementsEvaluator:
    """Independent evaluator for statement presence in generated code."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        code = state.get("final_code") or state.get("generated_code") or ""
        statements = self.task_specs.get(task_id, {}).get("correct_statements", [])

        status_items = await self.evaluator.eval_context_statements(
            context=code,
            statements=statements,
            judge_model_name=self.judge_model_name,
        )

        score, present_count = _present_score(status_items)

        return {
            "key": "code_statements_score",
            "score": score,
            "comment": f"task={task_id} present={present_count}/{len(status_items)}",
            "metadata": {"raw": status_items},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


class CodeExecutionEvaluator:
    """Independent evaluator for generated code execution pass/fail."""

    def __init__(self, evaluator: Evaluator):
        self.evaluator = evaluator

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        code = state.get("final_code") or state.get("generated_code") or ""

        result = await self.evaluator.eval_code_exec(code)
        execution_pass = bool(result.get("pass", False))

        return {
            "key": "code_execution_score",
            "score": 1.0 if execution_pass else 0.0,
            "comment": f"task={task_id} exec={execution_pass}",
            "metadata": {"raw": result},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


# Dummy target: we evaluate pre-computed states only
def dummy_target(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"status": "precomputed", "task_id": inputs.get("task_id", "")}


rag_eval = RagStatementsEvaluator(core_evaluator, eval_tasks, judge_model_name=JUDGE_MODEL_NAME)
code_statements_eval = CodeStatementsEvaluator(core_evaluator, eval_tasks, judge_model_name=JUDGE_MODEL_NAME)
code_execution_eval = CodeExecutionEvaluator(core_evaluator)

In [5]:
eval_tasks['add_task']

{'task': 'Add a new task into my Notion Tasks database with the title "Add_task_test", date today, importance of 4 and project ID "31bcb17dcc4480dcb042f86e6a70bb70".',
 'properties': {'title': 'Add_task_test',
  'date': 'today',
  'importance': 4,
  'project_id': '31bcb17dcc4480dcb042f86e6a70bb70'},
 'solution': 'def add_new_task(database_id, title, importance, project_id):\n    import requests\n    import os\n    from datetime import datetime\n    from dotenv import load_dotenv\n    \n    load_dotenv()\n    NOTION_TOKEN = os.getenv("NOTION_TOKEN")\n    HEADERS = {\n        "Authorization": f"Bearer {NOTION_TOKEN}",\n        "Content-Type": "application/json",\n        "Notion-Version": "2022-06-28"\n    }\n    \n    url = "https://api.notion.com/v1/pages"\n    \n    # Parse "today" dynamically into an ISO 8601 string (YYYY-MM-DD)\n    today_date = datetime.now().strftime("%Y-%m-%d")\n    \n    # Schema for creating a new database page with specific property types\n    payload = {\n   

## Samples

In [31]:
# Flexible sample loading + dataset creation in a single cell
# Change these variables directly for each run
THREAD_PREFIX_FILTERS = []  # [] means no filtering
N_LAST_RUNS = 6

static_params = StaticParams()

states_to_evaluate: List[Dict[str, Any]] = []

# Create or fetch dataset
existing = None
for ds in client.list_datasets(dataset_name=DATASET_NAME):
    if ds.name == DATASET_NAME:
        existing = ds
        break
    
if existing:
    print(f"Found existing dataset with name {DATASET_NAME}. To avoid confusion, the code is set to raise an error if the dataset already exists. Please change the DATASET_NAME variable or remove the existing dataset to proceed.")
    raise ValueError("Dataset already exists. Please change DATASET_NAME or remove the existing dataset to proceed.")

dataset = client.create_dataset(dataset_name=DATASET_NAME)


async with AsyncSqliteSaver.from_conn_string(static_params.sqlite_saver_path) as checkpointer:
    # Gather unique thread IDs from checkpoint store
    thread_ids: List[str] = []
    seen_thread_ids = set()

    async for checkpoint in checkpointer.alist(None):
        thread_id = checkpoint.config.get("configurable", {}).get("thread_id")

        if not thread_id or thread_id in seen_thread_ids:
            continue

        if THREAD_PREFIX_FILTERS and not any(thread_id.startswith(prefix + "_") for prefix in THREAD_PREFIX_FILTERS):
            continue

        seen_thread_ids.add(thread_id)
        thread_ids.append(thread_id)

    # Most recent are generally appended later; keep this direct and easy to tweak
    selected_thread_ids = thread_ids[:N_LAST_RUNS]

    for thread_id in selected_thread_ids:
        config = {"configurable": {"thread_id": thread_id}}
        checkpoint_tuple = await checkpointer.aget_tuple(config)
        if not checkpoint_tuple:
            continue

        channel_values = checkpoint_tuple.checkpoint.get("channel_values", {})
        task_id = thread_id.rsplit("_", 1)[0] if "_" in thread_id else thread_id

        states_to_evaluate.append({
            "thread_id": thread_id,
            "task_id": task_id,
            "query": channel_values.get("user_prompt", ""),
            "pre_computed_state": channel_values,
        })

# Push examples
for state in states_to_evaluate:
    client.create_example(
        inputs={
            "query": state["query"],
            "task_id": state["task_id"],
            "thread_id": state["thread_id"],
        },
        outputs={
            "task_id": state["task_id"],
            "pre_computed_state": state["pre_computed_state"],
        },
        dataset_id=dataset.id,
    )

print(f"Dataset: {dataset.name} | uploaded examples: {len(states_to_evaluate)}")
states_to_evaluate[:2]

Dataset: Eval_Run_Batch_1 | uploaded examples: 6


[{'thread_id': 'update_task_status_synthetic_20260307131627',
  'task_id': 'update_task_status_synthetic',
  'query': 'Update the task with id 31acb17dcc4480edbe0bcc0079323783 by changing its "Status" to "Done" and clearing the date.',
  'pre_computed_state': {'user_prompt': 'Update the task with id 31acb17dcc4480edbe0bcc0079323783 by changing its "Status" to "Done" and clearing the date.',
   'retrieval_context': '## Coding Guidance: Updating Notion Data Source Properties\n\nHere\'s a concise summary for coding, based on the provided context:\n\n* **Properties represent columns:** Notion database columns are represented as "properties" in the API.\n* **`properties` object:** Updates are made via the `properties` object in the API request.\n* **Property Schema:** The structure of `properties` mirrors the initial schema used when creating the database (`initial_data_source[properties]`). Refer to the [Property object documentation](/reference/property-object) for details.\n* **`key` & `

In [32]:
for st in states_to_evaluate:
    print(f"Thread: {st['thread_id']} | Task: {st['task_id']} | Query: {st['query'][:50]}...")

Thread: update_task_status_synthetic_20260307131627 | Task: update_task_status_synthetic | Query: Update the task with id 31acb17dcc4480edbe0bcc0079...
Thread: retrieve_tasks_20260307131325 | Task: retrieve_tasks | Query: Retrieve the last 3 tasks from my Notion Tasks dat...
Thread: get_page_content_synthetic_20260307131057 | Task: get_page_content_synthetic | Query: Retrieve all the content and blocks from the inbox...
Thread: append_checklist_to_page_synthetic_20260307130844 | Task: append_checklist_to_page_synthetic | Query: Add a checklist to the inbox page containing the i...
Thread: add_toggle_to_page_20260307130801 | Task: add_toggle_to_page | Query: Add a new toggle to my Notion page with the title ...
Thread: add_task_20260307130715 | Task: add_task | Query: Add a new task into my Notion Tasks database with ...


## Evaluate

In [55]:
results = evaluate(
    dummy_target,
    data=DATASET_NAME,
    evaluators=[rag_eval, code_statements_eval, code_execution_eval],
    experiment_prefix=EXPERIMENT_PREFIX,
)
results

View the evaluation results for experiment: 'Offline-Notion-Eval-3ac31cba' at:
https://smith.langchain.com/o/67129af7-abb6-42fb-b044-bbe6030db476/datasets/728e6351-27b8-4353-8ba6-323a15d614f3/compare?selectedSessions=b417d726-5bcb-4111-a1a1-c626227a05c5




0it [00:00, ?it/s]

2026-03-08 13:58:03 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-08 13:58:14 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


...

2026-03-08 13:59:36 | INFO     | src.evaluator | eval_code_exec | pass=False
2026-03-08 13:59:44 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-08 13:59:55 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-08 13:59:55 | INFO     | src.evaluator | eval_code_exec | pass=True


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


,inputs.query,inputs.task_id,inputs.thread_id,outputs.status,outputs.task_id,error,reference.task_id,reference.pre_computed_state,feedback.rag_statements_score,feedback.code_statements_score,feedback.code_execution_score,execution_time,example_id,id
0,Add a new task into my Notion Tasks database w...,add_task,add_task_20260307130715,precomputed,add_task,None,add_task,"{'passed': False, 'trials': [], 'verdict': {},...",0.000000,0.333333,1.0,0.006565,0bd45fb8-8be6-42f9-bb03-eb23fd3a5db5,019ccd18-b525-7b71-a249-fcae49a74aa8
1,Add a new toggle to my Notion page with the ti...,add_toggle_to_page,add_toggle_to_page_20260307130801,precomputed,add_toggle_to_page,None,add_toggle_to_page,"{'passed': False, 'trials': [], 'verdict': {},...",0.000000,0.833333,1.0,0.000382,c071cbc0-d18f-4f61-9003-2ce0a6fc3fff,019ccd19-0684-79b0-a9db-72fa94c077a8
2,Add a checklist to the inbox page containing t...,append_checklist_to_page_synthetic,append_checklist_to_page_synthetic_20260307130844,precomputed,append_checklist_to_page_synthetic,None,append_checklist_to_page_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",0.166667,0.500000,1.0,0.000660,00c61863-ceee-41b7-b363-89a442a11c16,019ccd19-518e-7423-a853-77d178b33d29
3,Retrieve all the content and blocks from the i...,get_page_content_synthetic,get_page_content_synthetic_20260307131057,precomputed,get_page_content_synthetic,None,get_page_content_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",1.000000,1.000000,1.0,0.000378,b26a15bb-1a07-4e7b-9b02-e65263c58d75,019ccd19-9c55-7433-8eea-e038840cc350
4,Retrieve the last 3 tasks from my Notion Tasks...,retrieve_tasks,retrieve_tasks_20260307131325,precomputed,retrieve_tasks,None,retrieve_tasks,"{'passed': False, 'trials': [], 'verdict': {},...",0.000000,0.666667,0.0,0.000633,7b49ce97-3c57-4879-b87c-0ca7d2324b84,019ccd19-ef83-7d12-a5f6-1ec08b4764c4
5,Update the task with id 31acb17dcc4480edbe0bcc...,update_task_status_synthetic,update_task_status_synthetic_20260307131627,precomputed,update_task_status_synthetic,None,update_task_status_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",0.333333,0.833333,1.0,0.000379,eb3ca776-68d7-4a5b-bdd0-c89c8161c192,019ccd1a-43bf-7d10-9b93-d65e3f40a6cb


In [53]:
for st in states_to_evaluate[1:2]:
    res = await core_evaluator.eval_context_statements(
        context=st["pre_computed_state"].get("generated_code", ""),
        statements=eval_tasks.get(st["task_id"], {}).get("correct_statements", []),
    )
    print(res)

2026-03-08 13:55:09 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
[{'statement': 'Endpoint is POST https://api.notion.com/v1/databases/{database_id}/query for querying a database.', 'status': 'Wrong', 'evidence': 'url = f"https://api.notion.com/v1/databases/query"', 'reasoning': 'The endpoint is missing the database_id path parameter.'}, {'statement': "Header 'Notion-Version: 2022-06-28' is required on every request.", 'status': 'Present', 'evidence': 'headers = {\n        "Authorization": f"Bearer {os.getenv(\'NOTION_TOKEN\')}",\n        "Notion-Version": "2022-06-28"\n    }', 'reasoning': "The header 'Notion-Version: 2022-06-28' is present in the headers dictionary."}, {'statement': "Result count is limited via the 'page_size' key in the request body (e.g. 'page_size': 3).", 'status': 'Present', 'evidence': '"page_size": num_tasks', 'reasoning': "The 'page_size' key is used in the request body to limit results."}, {'statement': "Sorting is defined under a 'sorts' array: [{'property': '